In [21]:
pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

Note: you may need to restart the kernel to use updated packages.


In [22]:
import os, time

print("Project ID found:", bool(os.environ.get("gcl_project_id")))
print("GCP service account key found:", bool(os.environ.get("GCP_SA_KEY")))

Project ID found: True
GCP service account key found: True


In [23]:
import sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = os.environ["gcl_project_id"]   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: datalab-504011


In [24]:
# Завдання 7. Ноутбук 00 — оркестратор

# Завдання 7.1. Встановіть papermill — стандартний інструмент для програмного запуску ноутбуків:

!pip install papermill

In [25]:
# Завдання 7.2. Задайте список ноутбуків у порядку виконання і теку для копій із результатами:

NOTEBOOKS = [
    "01_bronze.ipynb",
    "02_dim_currency.ipynb",
    "03_dim_date.ipynb",
    "04_snapshot.ipynb",
    "05_fact_merge.ipynb",
]

os.makedirs("runs", exist_ok=True)



In [26]:
import os
import sys
import json
import uuid
import pandas as pd
import papermill as pm

NOTEBOOKS = [
    "01_bronze.ipynb",
    "02_dim_currency.ipynb",
    "03_dim_date.ipynb",
    "04_snapshot.ipynb",
    "05_fact_merge.ipynb",
]

os.makedirs("runs", exist_ok=True)

# Generate a single UUID4 for the entire orchestration run
run_id = str(uuid.uuid4())
logs = []

# Loop through notebooks with a step counter starting at 1
for step_no, name in enumerate(NOTEBOOKS, start=1):
    started = pd.Timestamp.now(tz="UTC")

    try:
        pm.execute_notebook(
            name,
            f"runs/{name}",
            kernel_name="python3"
        )
        status, error = "OK", None

    except Exception as e:
        status = "FAILED"
        error = f"{type(e).__name__}: {e}"[:1000]

    finished = pd.Timestamp.now(tz="UTC")
    duration_sec = (finished - started).total_seconds()

    logs.append({
        "run_id": run_id,
        "step_no": step_no,
        "notebook": name,
        "status": status,
        "started_at": started,
        "finished_at": finished,
        "duration_sec": duration_sec,
        "error": error
    })

    if status == "FAILED":
        print(f"Orchestrator is failed! The problem with the notebook: {name}")
        break

# Create final DataFrame matching all Task 7.5 schema requirements
run_log = pd.DataFrame(logs)
run_log

,run_id,step_no,notebook,status,started_at,finished_at,duration_sec,error
0,348530ad-293c-4863-927c-8d2669a5f914,1,01_bronze.ipynb,OK,2026-09-01 11:57:09.020696+00:00,2026-09-01 11:57:18.806765+00:00,9.786069,None
1,348530ad-293c-4863-927c-8d2669a5f914,2,02_dim_currency.ipynb,OK,2026-09-01 11:57:18.806847+00:00,2026-09-01 11:57:28.470125+00:00,9.663278,None
2,348530ad-293c-4863-927c-8d2669a5f914,3,03_dim_date.ipynb,OK,2026-09-01 11:57:28.470184+00:00,2026-09-01 11:57:38.620836+00:00,10.150652,None
3,348530ad-293c-4863-927c-8d2669a5f914,4,04_snapshot.ipynb,OK,2026-09-01 11:57:38.620898+00:00,2026-09-01 11:57:47.659111+00:00,9.038213,None
4,348530ad-293c-4863-927c-8d2669a5f914,5,05_fact_merge.ipynb,OK,2026-09-01 11:57:47.659190+00:00,2026-09-01 11:58:00.193241+00:00,12.534051,None


In [27]:
# Створіть датасет nbu_meta і таблицю nbu_meta.run_log із партиціюванням за датою з started_at, запишіть журнал у режимі WRITE_APPEND.

ds = bigquery.Dataset(f"{PROJECT_ID}.nbu_meta")
ds.location = "EU"
client.create_dataset(ds, exists_ok=True)



# Завдання 2.2. Створіть таблицю

nbu_run_log = f"{PROJECT_ID}.nbu_meta.run_log"

schema = [
    bigquery.SchemaField("run_id", "STRING"),
    bigquery.SchemaField("step_no", "INTEGER"),
    bigquery.SchemaField("notebook", "STRING"),
    bigquery.SchemaField("status", "STRING"),
    bigquery.SchemaField("started_at", "TIMESTAMP"),
    bigquery.SchemaField("finished_at", "TIMESTAMP"),
    bigquery.SchemaField("duration_sec", "FLOAT"),
    bigquery.SchemaField("error", "STRING"),
]

table = bigquery.Table(nbu_run_log, schema=schema)
table.time_partitioning = bigquery.TimePartitioning(field="started_at")

table = client.create_table(table, exists_ok=True)


cfg = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND
)


job = client.load_table_from_dataframe(run_log, nbu_run_log, job_config=cfg)
job.result()


print("Saved log data to BigQuery", nbu_run_log)


Saved log data to BigQuery datalab-504011.nbu_meta.run_log


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [33]:
# Завдання 7.7. Виведіть підсумок прогону: скільки ноутбуків завершилося успішно, скільки впало, загальна тривалість. 

print(f"Succesfully run notebooks number: {(run_log['status'] == 'OK').sum()}")
print(f"Failed notebooks number: {(run_log['status'] == 'FAILED').sum()}")
print("Total run time", run_log["duration_sec"].sum())

query = f"""
SELECT run_id, step_no, notebook, status, duration_sec
FROM `{PROJECT_ID}.nbu_meta.run_log`
ORDER BY started_at DESC
LIMIT 10
"""

notebooks_data = client.query(query).to_dataframe()
notebooks_data

Succesfully run notebooks number: 5
Failed notebooks number: 0
Total run time 51.172263


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,run_id,step_no,notebook,status,duration_sec
0,348530ad-293c-4863-927c-8d2669a5f914,5,05_fact_merge.ipynb,OK,12.534051
1,348530ad-293c-4863-927c-8d2669a5f914,4,04_snapshot.ipynb,OK,9.038213
2,348530ad-293c-4863-927c-8d2669a5f914,3,03_dim_date.ipynb,OK,10.150652
3,348530ad-293c-4863-927c-8d2669a5f914,2,02_dim_currency.ipynb,OK,9.663278
4,348530ad-293c-4863-927c-8d2669a5f914,1,01_bronze.ipynb,OK,9.786069
5,e60a4dde-31c6-4d1e-82b7-5f7ff03d5d40,5,05_fact_merge.ipynb,OK,16.201250
6,e60a4dde-31c6-4d1e-82b7-5f7ff03d5d40,4,04_snapshot.ipynb,OK,10.205765
7,e60a4dde-31c6-4d1e-82b7-5f7ff03d5d40,3,03_dim_date.ipynb,OK,8.143732
8,e60a4dde-31c6-4d1e-82b7-5f7ff03d5d40,2,02_dim_currency.ipynb,OK,11.214978
9,e60a4dde-31c6-4d1e-82b7-5f7ff03d5d40,1,01_bronze.ipynb,OK,12.306519
